In [ ]:
import pandas as pd

In [ ]:
df3 = pd.read_pickle("df3.pkl")

Getting planning area dataset
- API is from https://www.onemap.gov.sg/apidocs/planningarea/#planningAreaPolygon for year 2019 which is the latest available year
- Please get your own token from One Map and insert it into headers and uncomment that line

In [ ]:
#please get your own token from one map api and insert into headers and uncomment headers. 
import requests
    
url = "https://www.onemap.gov.sg/api/public/popapi/getAllPlanningarea?year=2019"
    
headers = {"Authorization": "***insert token***"}
    
response = requests.request("GET", url, headers=headers)
    
print(response.text)

data = response.json()

In [ ]:
records = data['SearchResults']
areas = pd.DataFrame(records)
import json
areas['geojson_parsed'] = areas['geojson'].apply(json.loads)

In [ ]:
from shapely.geometry import shape
import geopandas as gpd

areas['geometry'] = areas['geojson_parsed'].apply(shape)

gdf = gpd.GeoDataFrame(areas, geometry='geometry', crs="EPSG:4326")

In [ ]:
gdf = gdf[['pln_area_n', 'geometry']]
gdf = gdf.rename(columns={'pln_area_n': 'planning_area'})

In [ ]:
def get_planning_area(df, lat_col, lon_col):
    gdf_points = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df[lon_col], df[lat_col]),
        crs="EPSG:4326"
    )
    return gpd.sjoin(gdf_points, gdf, how="left", predicate="within")['planning_area'].values

In [ ]:
first_tap = df3.groupby('CRD_NUM', as_index=False).first()
first_tap['house_area'] = get_planning_area(first_tap, 'ORIG_LATITUDE', 'ORIG_LONGITUDE')

df3 = df3.merge(first_tap[['CRD_NUM', 'house_area']], on='CRD_NUM', how='left')

In [ ]:
# find origin and destination for every row
df3['orig_region'] = get_planning_area(df3, 'ORIG_LATITUDE', 'ORIG_LONGITUDE')
df3['dest_region'] = get_planning_area(df3, 'DEST_LATITUDE', 'DEST_LONGITUDE')

In [ ]:
# check distribution of regions
print(df3['orig_region'].value_counts())
print(df3['dest_region'].value_counts())

In [ ]:
# check house area: each card should have exactly one house_area
print(df3.groupby('CRD_NUM')['house_area'].nunique().value_counts())
# 1 means every card has exactly 1 house area
# 0 means card has no house area -> likely because their first tap-in is at woodlands checkpoint
#should not have any values above 1

Verification
- orig_region, dest_region, and house_area should be NaN when the area is woodlands checkpoint/jb

In [ ]:
# Rows where ORIG_REGION is NA: show CRD_NUM, ORIG_STATION_NAME
orig_region_na = df3[df3['orig_region'].isna()][
    ['CRD_NUM', 'ORIG_STATION_NAME', 'ORIG_LATITUDE', 'ORIG_LONGITUDE']
]
#print("Rows with missing orig_region:")
#print(orig_region_na.head(20))

# Rows where DEST_REGION is NA: show DEST_STATION_NAME
dest_region_na = df3[df3['dest_region'].isna()][
    ['CRD_NUM', 'DEST_STATION_NAME', 'DEST_LATITUDE', 'DEST_LONGITUDE']
]
#print("\nRows with missing dest_region:")
#print(dest_region_na.head(20))


In [ ]:
# Rows where HOUSE_AREA is NaN: show CRD_NUM, ORIG_STATION_NAME
house_area_na = df3[df3['house_area'].isna()][
    ['CRD_NUM', 'ORIG_STATION_NAME', 'ORIG_LATITUDE', 'ORIG_LONGITUDE']
]
#print("\nRows with missing house_area:")
#print(house_area_na.head(20))

In [ ]:
# Save df3 as a pickle
df3.to_pickle("df3_regions.pkl")

In [ ]:
# Save df3 as a CSV file
#df3.to_csv("df_regions.csv", index=False)